In [1]:
# Cell 1: Install necessary libraries
!pip install transformers scikit-learn pandas torch sentencepiece

In [2]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm # For nice progress bars
import os

In [6]:
# Cell 3: Configuration

# --- (FIXED) Configuration ---
TRAIN_FILE = 'healthver_train.csv'
T5_MODEL = 'Vamsi/T5_Paraphrase_Paws' # <-- This is the new, correct model
AUGMENTED_FILE_NAME = 'healthver_train_augmented.csv' # Our new output file

TARGET_REFUTES_SAMPLES = 3500 # Your teacher's target

In [7]:
# Cell 4: Load Data and Prepare Labels

print("Loading data...")
print(f"IMPORTANT: Make sure you have uploaded {TRAIN_FILE} to this session!")

try:
    df_train = pd.read_csv(TRAIN_FILE)
    print("Data loaded successfully.")
    print("\nOriginal label counts:")
    print(df_train['label'].value_counts())

except FileNotFoundError:
    print(f"Error: Could not find {TRAIN_FILE}.")
    print("Please upload your CSV file and re-run this cell.")

Loading data...
IMPORTANT: Make sure you have uploaded healthver_train.csv to this session!
Data loaded successfully.

Original label counts:
label
Neutral     4397
Supports    3782
Refutes     2411
Name: count, dtype: int64


In [8]:
# Cell 5: Load T5 Paraphrasing Model

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print(f"Loading tokenizer for {T5_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)

print(f"Loading model {T5_MODEL}...")
model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL).to(device)
model.eval() # Set to evaluation mode
print("Paraphrasing model loaded successfully.")

Using device: cuda
Loading tokenizer for Vamsi/T5_Paraphrase_Paws...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loading model Vamsi/T5_Paraphrase_Paws...


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Paraphrasing model loaded successfully.


In [9]:
# Cell 6: Define Paraphrase Function

def paraphrase_claim(text, model, tokenizer):
    """
    Paraphrases a single piece of text using the T5 model.
    """
    # T5 models expect a prefix for the task
    text_to_paraphrase = "paraphrase: " + text + " </s>"

    # Tokenize
    encoding = tokenizer(
        text_to_paraphrase,
        padding='longest',
        max_length=128,
        truncation=True,
        return_tensors="pt"
    )

    input_ids = encoding.input_ids.to(device)
    attention_mask = encoding.attention_mask.to(device)

    # Generate the new text
    # We use beam search for better results
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=5,          # 5 beams for beam search
            num_return_sequences=1, # We just want the best one
            early_stopping=True
        )

    # Decode the new text
    new_claim = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return new_claim

In [10]:
# Cell 7: Generate New 'Refutes' Samples

if 'df_train' in locals():
    print("Starting data augmentation for 'Refutes' class...")

    # 1. Isolate the original 'Refutes' samples
    refutes_df = df_train[df_train['label'] == 'Refutes'].copy()
    num_original_refutes = len(refutes_df)

    # 2. Calculate how many new samples we need
    num_to_generate = TARGET_REFUTES_SAMPLES - num_original_refutes

    if num_to_generate <= 0:
        print(f"You already have {num_original_refutes} 'Refutes' samples, which meets the target of {TARGET_REFUTES_SAMPLES}.")
        print("No new samples will be generated.")
    else:
        print(f"Original 'Refutes' count: {num_original_refutes}")
        print(f"Target count: {TARGET_REFUTES_SAMPLES}")
        print(f"Need to generate: {num_to_generate} new samples.")

        # 3. Select claims to paraphrase. We will sample *with replacement*
        # This means we might paraphrase the same claim more than once, which is fine.
        samples_to_paraphrase = refutes_df.sample(num_to_generate, replace=True)

        new_rows = []

        # 4. Loop and generate
        print("Starting paraphrase generation... This will take time.")
        for index, row in tqdm(samples_to_paraphrase.iterrows(), total=num_to_generate):
            original_claim = row['claim']

            # Generate the new claim
            try:
                new_claim = paraphrase_claim(original_claim, model, tokenizer)

                # Copy the original row
                new_row = row.copy()
                # Set the new paraphrased claim
                new_row['claim'] = new_claim
                # We can also update the 'id' to show it's augmented
                new_row['id'] = f"{row['id']}_aug"

                new_rows.append(new_row)
            except Exception as e:
                print(f"Error paraphrasing claim at index {index}: {e}")

        print(f"Successfully generated {len(new_rows)} new samples.")

        # 5. Create new DataFrames and save
        if len(new_rows) > 0:
            new_samples_df = pd.DataFrame(new_rows)

            # 6. Combine original data + new samples
            augmented_df = pd.concat([df_train, new_samples_df], ignore_index=True)

            print(f"\nSuccessfully created new augmented DataFrame.")
            print(f"Total rows in new file: {len(augmented_df)}")

            print("\nNew label counts:")
            print(augmented_df['label'].value_counts())

            # 7. Save the new file
            augmented_df.to_csv(AUGMENTED_FILE_NAME, index=False)
            print(f"\nSuccessfully saved new dataset to '{AUGMENTED_FILE_NAME}'")
            print(f"You can now download this file and use it for the (New) Phase 5.")
        else:
            print("No new rows were generated. File not saved.")
else:
    print("df_train not loaded. Please run Cell 4 successfully first.")

Starting data augmentation for 'Refutes' class...
Original 'Refutes' count: 2411
Target count: 3500
Need to generate: 1089 new samples.
Starting paraphrase generation... This will take time.


  0%|          | 0/1089 [00:00<?, ?it/s]

Successfully generated 1089 new samples.

Successfully created new augmented DataFrame.
Total rows in new file: 11679

New label counts:
label
Neutral     4397
Supports    3782
Refutes     3500
Name: count, dtype: int64

Successfully saved new dataset to 'healthver_train_augmented.csv'
You can now download this file and use it for the (New) Phase 5.
